In [1]:
import csv

In [2]:
path_ob = '/home/rafaeljpd/Data/cimetrias/correction-bases/v0.6c/base_issnl2all_v0.6c.csv'
path_nb = '/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/issn_to_all.v0.8.csv'

In [3]:
gissn_to_data = {}
issn_to_gissn = {}

with open(path_nb) as fin:
    for row in csv.DictReader(fin, fieldnames=fin.readline().strip().split('|'), delimiter='|'):
        gissn = row['GOLD_ISSN']
        issns = row['ISSNS'].split('#')
        for i in issns:
            issn_to_gissn[i] = gissn
        gissn_to_data[gissn] = row

In [4]:
gissn_to_data['0000-0027']

{'GOLD_ISSN': '0000-0027',
 'ISSNS': '0000-0027#2767-5505',
 'PORTAL_ISSN_2019_MAIN_TITLE': 'LIBRARY JOURNAL',
 'PORTAL_ISSN_2019_MAIN_ABBREVIATED_TITLE': 'LIBR J 1876',
 'PORTAL_ISSN_2024_MAIN_TITLE': 'LIBRARY JOURNAL',
 'TITLES': 'LIBRARY JOURNAL 1876 ONLINE#LIBRARY JOURNAL 1876#LIBRARY JOURNAL#LIBR J 1876#AMERICAN LIBRARY JOURNAL',
 'COUNTRIES': 'UNITED STATES#NEW YORK STATE',
 'WOS_JCR_COUNTRIES': '',
 'WOS_EXTRA_COUNTRIES': '',
 'SCIELO_COUNTRIES': '',
 'LATINDEX_COUNTRIES': '',
 'PORTAL_ISSN_2024_COUNTRIES': 'NEW YORK STATE',
 'PORTAL_ISSN_2019_COUNTRIES': 'UNITED STATES',
 'IS_DOAJ': '0',
 'IS_LATINDEX': '0',
 'IS_MS_BRAZIL': '0',
 'IS_MS_SPAIN': '0',
 'IS_MI_EXTRA': '0',
 'IS_NLM': '0',
 'IS_PORTAL_ISSN_2019': '1',
 'IS_PORTAL_ISSN_2024': '1',
 'IS_SCIELO': '0',
 'IS_SCIMAGOJR': '0',
 'IS_SCOPUS_ACCEPTED': '0',
 'IS_SCOPUS_SOURCES': '0',
 'IS_ULRICH': '0',
 'IS_WOS_EXTRA': '0',
 'IS_WOS_JCR': '0'}

In [5]:
old_gissn_to_data = {}
old_issn_to_gissn = {}

with open(path_ob) as fin:
    for row in csv.DictReader(fin, fieldnames=fin.readline().strip().split('|'), delimiter='|'):
        gissn = row['ISSNL']
        issns = row['ISSNs'].split('#')
        for i in issns:
            old_issn_to_gissn[i] = gissn
        old_gissn_to_data[gissn] = row

In [6]:
old_gissn_to_data[issn_to_gissn['2178-938X']]

{'ISSNL': '0034-7590',
 'MAIN_TITLE': 'RAE',
 'MAIN_ABBREV_TITLE': 'RAE IMPR#RAE',
 'ISSNs': '2178-938X#0034-7590',
 'OTHER_TITLEs': 'REV ADM EMPRES#RAE REV ADMIN EMPRES#REVISTA DE ADMINISTRACAO DE EMPRESAS#RAE IMPR#RAE REV ADM EMPRESAS#RAE#RAE REVISTA DE ADMINISTRACAO DE EMPRESAS',
 'PORTAL_ISSN': '1',
 'DOAJ': '1',
 'LATINDEX': '1',
 'NLM': '0',
 'SCIELO': '1',
 'SCIMAGO_JR': '1',
 'SCOPUS': '1',
 'ULRICH': '1',
 'WOS': '1',
 'WOS_JCR': '1',
 'COUNTRIES': '0034-7590-BRAZIL#2178-938X-BRAZIL',
 'YEARS': '0034-7590-1961-2017#2178-938X-1961-'}

In [7]:
richs = 0
missing = 0
dups = 0

for oi in old_gissn_to_data:
    o_titles = []
    o_titles.extend(old_gissn_to_data[oi]['MAIN_TITLE'].split('#'))
    o_titles.extend(old_gissn_to_data[oi]['MAIN_ABBREV_TITLE'].split('#'))
    o_titles.extend(old_gissn_to_data[oi]['OTHER_TITLEs'].split('#'))

    o_titles = sorted(set([t for t in o_titles if t != '']))
    
    issns = []
    issns.extend(old_gissn_to_data[oi]['ISSNs'].split('#'))
    issns.append(old_gissn_to_data[oi]['ISSNL'])

    gis = set()
    for i in issns:
        gi = issn_to_gissn.get(i)
        if gi:
            gis.add(gi)

    if len(gis) == 0:
        missing += 1

    if len(gis) > 1:
        dups += 1
    
    if len(gis) == 1:
        gissn = gis.pop()

        n_titles = [
            gissn_to_data[gissn]['PORTAL_ISSN_2019_MAIN_TITLE'],
            gissn_to_data[gissn]['PORTAL_ISSN_2019_MAIN_ABBREVIATED_TITLE'],
            gissn_to_data[gissn]['PORTAL_ISSN_2024_MAIN_TITLE'],
        ]
        n_titles.extend(gissn_to_data[gissn]['TITLES'].split('#'))
        n_titles = sorted(set([t for t in n_titles if t != '']))
        
        o_titles = set(o_titles)
        n_titles = set(n_titles)
        
        if not n_titles.issuperset(o_titles):
            richs += 1

        gissn_to_data[gissn]['TITLES'] = '#'.join(sorted(set(n_titles.union(o_titles))))

print(richs, missing, dups)

70127 35 369


In [8]:
for g in gissn_to_data:
    gissn_to_data[g]['TITLES'] = '#'.join(sorted(set(gissn_to_data[g]['TITLES'].split('#'))))

In [9]:
with open('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/issn_to_all.v0.8b.csv', 'w') as fout:
    header = [
        'gold_issn',
        'issns',
        'portal_issn_2019_main_title',
        'portal_issn_2019_main_abbreviated_title',
        'portal_issn_2024_main_title',
        'titles',
        'countries',
        'wos_jcr_countries',
        'wos_extra_countries',
        'scielo_countries',
        'latindex_countries',
        'portal_issn_2024_countries',
        'portal_issn_2019_countries',
        'is_doaj',
        'is_latindex',
        'is_ms_brazil',
        'is_ms_spain',
        'is_mi_extra',
        'is_nlm',
        'is_portal_issn_2019',
        'is_portal_issn_2024',
        'is_scielo',
        'is_scimagojr',
        'is_scopus_accepted',
        'is_scopus_sources',
        'is_ulrich',
        'is_wos_extra',
        'is_wos_jcr'
    ]

    fout.write('|'.join(header).upper() + '\n')
    for i in gissn_to_data:
        fout.write('|'.join([gissn_to_data[i][k.upper()] for k in header]) + '\n')

In [10]:
gissn_to_data[issn_to_gissn['2178-938X']]

{'GOLD_ISSN': '0034-7590',
 'ISSNS': '0034-7590#2178-938X',
 'PORTAL_ISSN_2019_MAIN_TITLE': 'RAE',
 'PORTAL_ISSN_2019_MAIN_ABBREVIATED_TITLE': 'RAE ONLINE#RAE IMPR',
 'PORTAL_ISSN_2024_MAIN_TITLE': 'RAE',
 'TITLES': 'FUNDACAO GETULIO VARGAS ESCOLA DE ADMINISTRACAO DE EMPRESAS DE S PAULO#RAE#RAE IMPR#RAE IMPRESSO#RAE ONLINE#RAE REV ADM EMPRESAS#RAE REV ADMIN EMPRES#RAE REVISTA DE ADMINISTRACAO DE EMPRESAS#REV ADM EMPRES#REVISTA DE ADMINISTRACAO DE EMPRESAS',
 'COUNTRIES': 'SCL#BRASIL#BRAZIL',
 'WOS_JCR_COUNTRIES': 'BRAZIL',
 'WOS_EXTRA_COUNTRIES': '',
 'SCIELO_COUNTRIES': 'SCL',
 'LATINDEX_COUNTRIES': 'BRASIL',
 'PORTAL_ISSN_2024_COUNTRIES': 'BRAZIL',
 'PORTAL_ISSN_2019_COUNTRIES': 'BRAZIL',
 'IS_DOAJ': '1',
 'IS_LATINDEX': '1',
 'IS_MS_BRAZIL': '1',
 'IS_MS_SPAIN': '0',
 'IS_MI_EXTRA': '0',
 'IS_NLM': '0',
 'IS_PORTAL_ISSN_2019': '1',
 'IS_PORTAL_ISSN_2024': '1',
 'IS_SCIELO': '1',
 'IS_SCIMAGOJR': '1',
 'IS_SCOPUS_ACCEPTED': '0',
 'IS_SCOPUS_SOURCES': '1',
 'IS_ULRICH': '1',
 'IS_WOS_